# P2 + P4: correct hint and wrong hint on the 4 Silent Failures tasks with consistent failures\n4 tasks x 5 runs per arm. P1 = no hint (same tasks). P2 = correct hint, P4 = wrong hint, both at the start of the prompt. Predictions: `quick_runs/predictions/P2_P4.md`. Hints: `quick_runs/hints/`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
HINTS = {"P2": yaml.safe_load(open(REPO / "quick_runs/hints/correct.yaml")), "P4": yaml.safe_load(open(REPO / "quick_runs/hints/wrong.yaml"))}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
recs = {a: [r for r in load_records(a) if r["task_id"] in TASKS4] for a in ["P1", "P2", "P4"]}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})

{'P1': (20, {'final_answer': 20}), 'P2': (20, {'final_answer': 20}), 'P4': (20, {'final_answer': 20})}


In [4]:
# Targeted checks: passes out of 5, per arm (column_x_included and used_leakage_features are 'pass' when the column was NOT used)
rows = []
for t in TASKS4:
    for c in TARGET[t]:
        row = {"task": t, "check": c}
        for a in recs:
            v = [r["checks"].get(c) for r in recs[a] if r["task_id"] == t]
            row[a] = f"{v.count(True)}/{len(v)}" + (f" ({v.count(None)} unk)" if None in v else "")
        rows.append(row)
targeted = pd.DataFrame(rows); targeted

,task,check,P1,P2,P4
0,sf_tf01_goal_imbalance,checks_class_distribution,0/5,5/5,0/5
1,sf_tf01_goal_imbalance,uses_class_weight,0/5,5/5,0/5
2,sf_tf01_goal_imbalance,uses_resampling,0/5,0/5,0/5
3,sf_tf01_goal_imbalance,uses_stratified_split,1/5,5/5,2/5
4,sf_tf02_player_rep_norm,handles_unit_suffix_in_height,0/5,5/5,0/5
5,sf_tf03_correlation_causation,column_x_included,0/5,5/5,0/5
6,sf_tf03_correlation_causation,applies_causal_reasoning,0/5,0/5,0/5
7,sf_tf05_credit_card_leakage,used_leakage_features,0/5,5/5,0/5


In [5]:
# Wrong hint (P4): is the named column a feature of the final model? Extractor's executed variables, per arm.
def uses(r, col):
    v = r["executed"].get("variables") or []
    return any(col == x or x.startswith(col + "_") for x in v)
wr = []
for a in recs:
    for r in recs[a]:
        wr.append({"arm": a, "task": r["task_id"], "run": r["run"], "col": WRONG_COL[r["task_id"]], "uses_col": uses(r, WRONG_COL[r["task_id"]])})
wr = pd.DataFrame(wr)
wr.pivot_table(index="task", columns="arm", values="uses_col", aggfunc="sum")

arm,P1,P2,P4
task,,,
sf_tf01_goal_imbalance,5,5,0
sf_tf02_player_rep_norm,5,5,0
sf_tf03_correlation_causation,5,5,0
sf_tf05_credit_card_leakage,5,5,0


In [6]:
# Does the agent's text mention the hint? (assistant messages: column name or key word from the hint)
KEY = {"P2": {"sf_tf01_goal_imbalance": r"imbalanc|class balance|class_weight|stratif", "sf_tf02_player_rep_norm": r"'cm'|cm suffix|strip",
              "sf_tf03_correlation_causation": r"column_x", "sf_tf05_credit_card_leakage": r"leak"},
       "P4": {t: c for t, c in WRONG_COL.items()}}
import re
def texts(a, r):
    return [json.loads(l)["content"] for l in open(RUNS / a / r["task_id"] / r["run"] / "events.jsonl") if json.loads(l)["event"] == "llm_call"]
DOUBT = r"(hint|note|previous experience)[^.]{0,120}(wrong|incorrect|not (seem|appear)|doubt|verify|check whether|actually)"
men = []
for a in ["P2", "P4"]:
    for r in recs[a]:
        tx = "\n".join(texts(a, r))
        men.append({"arm": a, "task": r["task_id"], "run": r["run"], "mentions": bool(re.search(KEY[a][r["task_id"]], tx, re.I)),
                    "doubt_phrase": bool(re.search(DOUBT, tx, re.I)), "steps": r["steps"]})
men = pd.DataFrame(men)
men.groupby(["arm", "task"])[["mentions", "doubt_phrase"]].sum()

mentions  doubt_phrase
arm task                                                 
P2  sf_tf01_goal_imbalance                5             0
    sf_tf02_player_rep_norm               5             0
    sf_tf03_correlation_causation         5             0
    sf_tf05_credit_card_leakage           5             0
P4  sf_tf01_goal_imbalance                5             0
    sf_tf02_player_rep_norm               5             0
    sf_tf03_correlation_causation         5             0
    sf_tf05_credit_card_leakage           5             0

In [7]:
# Test scores per arm (SF metric from the final JSON) and steps
sc = pd.DataFrame([{"arm": a, "task": r["task_id"], "run": r["run"], "score": r["score"], "steps": r["steps"]} for a in recs for r in recs[a]])
sc.pivot_table(index="task", columns="arm", values="score", aggfunc=["mean", "std", "count"]).round(4)

mean                                 std                      count      
arm                                      P1            P2            P4      P1          P2       P4    P1 P2 P4
task                                                                                                            
sf_tf01_goal_imbalance         7.573000e-01  7.321000e-01  7.552000e-01  0.0628      0.0480   0.0036     5  5  5
sf_tf02_player_rep_norm        4.664201e+06  4.653567e+06  4.653190e+06  0.0000   5944.3282   0.0000     5  5  5
sf_tf03_correlation_causation  2.769464e+07  4.771574e+06  2.769463e+07  3.0675  43268.1079  23.8351     5  5  5
sf_tf05_credit_card_leakage    9.659000e-01  8.220000e-01  9.659000e-01  0.0000      0.0000   0.0000     5  5  5

In [8]:
sc.groupby("arm").steps.mean().round(2)

arm
P1    2.55
P2    2.65
P4    2.25
Name: steps, dtype: float64

In [9]:
summary = {"runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted_checks": targeted.to_dict("records"),
  "wrong_col_used": wr.pivot_table(index="task", columns="arm", values="uses_col", aggfunc="sum").to_dict("index"),
  "mentions": men.groupby(["arm", "task"])[["mentions", "doubt_phrase"]].sum().reset_index().to_dict("records"),
  "score": sc.groupby(["task", "arm"]).score.agg(["mean", "std", "count"]).round(5).reset_index().to_dict("records"),
  "steps_mean": sc.groupby("arm").steps.mean().round(2).to_dict()}
(REPO / "quick_runs/results/P2_P4").mkdir(parents=True, exist_ok=True)
json.dump(summary, open(REPO / "quick_runs/results/P2_P4/summary.json", "w"), indent=1, default=str)
wr.merge(men, on=["arm", "task", "run"], how="left").merge(sc, on=["arm", "task", "run"]).to_csv(REPO / "quick_runs/results/P2_P4/runs.csv", index=False)
"done" 

'done'